# 気象庁の公式データに、日本語で質問できる窓口
## — LLM の「Tool Use」を Google Colab で体験する —

このノートブックは、**LLM(大規模言語モデル)が気象庁の公開データを自分で取りに行って答える**仕組みを、
無料の Google Colab 上で動かしながら体験するためのものです。

- 使う LLM は **Qwen3**(Alibaba Cloud が公開しているオープンなモデル)。Colab の GPU 上でローカルに動かすので、API キーも課金も不要です
- データは **気象庁のウェブサイトが公開している JSON** だけを使います(週間予報・3 日間予報・概況文・警報注意報・アメダス実況)
- 「Tool」とは、LLM が呼び出せる **ただの Python 関数** です。関数を 1 つ書いて登録するだけで、LLM ができることが増えます

### 進め方
1. 上のメニュー **「ランタイム」→「ランタイムのタイプを変更」→ T4 GPU** を選ぶ(無料枠で OK)
2. 上から順にセルを実行する(▶ ボタン、または Shift+Enter)。モデルの読み込みに 5〜6 分かかります
3. 途中の「質問」セルは、文章を自由に書き換えて試してください

### 構成
| # | 内容 | ここで分かること |
|---|---|---|
| 1 | モデルを選ぶ | 自分の GPU に合うモデル |
| 2 | LLM の読み込み | (待ち時間) |
| 3 | **LLM 単体に聞く** | LLM は最新の天気を知らない |
| 4 | **Tool を 1 つ渡す** | 同じ質問に正しく答えるようになる |
| 5 | **Tool を増やす** | 複数の Tool を組み合わせて答える |
| 6 | 質問例いろいろ | 分野別の使い方 |
| 7 | 記事・お知らせ文を作って保存 | Markdown で表示、.md / .html でダウンロード |
| 8 | **落とし穴** | LLM は Tool の間違いを隠す |
| 9 | 自分の Tool を書いてみる | 関数 1 つで機能が増える |

> **データの出典と注意**: 気象データは気象庁ホームページ(https://www.jma.go.jp/)の公開データを利用しています。
> 本ノートブックは学習・デモ目的です。防災上の判断は必ず気象庁および自治体の公式発表に基づいてください。

## 1. モデルを選ぶ

Qwen3 にはサイズ違いがあります。**無料枠の T4(VRAM 16 GB)なら 8B が標準**です。
Colab Pro などで A100 が使える人は、30B-A3B(MoE: 実際に動く部分は 3B なので速くて賢い)も選べます。

| モデル | 量子化 | VRAM 目安 | T4 (16 GB) | A100 (40 GB) | 位置づけ |
|---|---|---|---|---|---|
| Qwen3-4B-Instruct-2507 | 4bit | 約 3 GB | ◎ 速い | ◎ | 軽量・時短(読み込み約半分) |
| **Qwen3-8B** | 4bit | 約 6 GB | ○ 標準 | ◎ | **デフォルト**。Tool 呼び出しの安定性と日本語のバランス |
| Qwen3-14B | 4bit | 約 10 GB | △ 遅い | ◎ | 品質重視 |
| Qwen3-30B-A3B-Instruct-2507 | 4bit | 約 18 GB | × | ◎ | A100 向け。MoE で速くて賢い |

下のセルのプルダウンで選んで実行すると、いま使える GPU と照らして載るかどうかを表示します。

In [ ]:
#@title モデルを選んで実行 → GPU との相性を確認
MODEL_NAME = "Qwen/Qwen3-8B" #@param ["Qwen/Qwen3-4B-Instruct-2507", "Qwen/Qwen3-8B", "Qwen/Qwen3-14B", "Qwen/Qwen3-30B-A3B-Instruct-2507"]

import torch
VRAM_NEED_GB = {  # 4bit 量子化時の目安(重み + 推論時の余裕)
    "Qwen/Qwen3-4B-Instruct-2507": 4,
    "Qwen/Qwen3-8B": 7,
    "Qwen/Qwen3-14B": 11,
    "Qwen/Qwen3-30B-A3B-Instruct-2507": 20,
}
if not torch.cuda.is_available():
    print("⚠️ GPU が見つかりません。メニュー「ランタイム」→「ランタイムのタイプを変更」で T4 GPU を選んでください。")
else:
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    need = VRAM_NEED_GB[MODEL_NAME]
    print(f"GPU: {gpu}  VRAM: {vram:.1f} GB")
    print(f"選択: {MODEL_NAME}  必要 VRAM 目安: 約 {need} GB")
    if vram >= need * 1.15:
        print("✅ このモデルは載ります")
    elif vram >= need:
        print("⚠️ ぎりぎりです。長い質問で GPU メモリ不足になったら、小さいモデルに変えてください")
    else:
        print("❌ この GPU には載りません。プルダウンで小さいモデルを選び直してください")

## 2. 準備 — ライブラリのインストールと LLM の読み込み

`transformers`(LLM を動かす)、`bitsandbytes`(4bit 量子化で VRAM を節約する)を入れて、モデルをダウンロードします。
**5〜6 分かかります**(8B の場合。ダウンロード約 16 GB)。その間に下の説明を読んでおいてください。

> **4bit 量子化とは**: モデルの重み(数十億個の数値)を 16bit から 4bit に圧縮して VRAM を 1/4 にする技術。精度はわずかに落ちますが、無料 GPU で 8B クラスが動くのはこのおかげです。

In [ ]:
#@title ライブラリのインストール(1〜2 分)
%pip install -q -U transformers bitsandbytes accelerate markdown
print("✅ インストール完了")

In [ ]:
#@title LLM の読み込み(5〜6 分)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 は bfloat16 非対応なので float16
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print(f"✅ 読み込み完了: {MODEL_NAME}")
print(f"   GPU メモリ使用量: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

### LLM に Tool を使わせる仕組み(`ask` 関数)

下のセルが、このノートブックの心臓部です。やっていることは単純で、

1. 質問と「使える関数の一覧」を LLM に渡す
2. LLM が `<tool_call>{"name": "関数名", "arguments": {...}}</tool_call>` と答えたら、その関数を **こちらで実行** して結果を LLM に返す
3. LLM が関数を呼ばずに普通の文章を返したら、それが最終回答

を、最大 6 往復まで繰り返すだけです。LLM 自身はインターネットに触れません。**関数を選び、引数を決め、結果を読んで文章にする**のが LLM の仕事で、**実際にデータを取ってくる**のは Python の仕事です。

「使える関数の一覧」は、Python 関数の **docstring(説明文)と型ヒント** から自動生成されます(Qwen3 のチャットテンプレートが対応)。だから関数を 1 つ書けば Tool が 1 つ増えます。

In [ ]:
#@title Tool Use の仕組み(ask 関数)
import re
import json
import datetime
import inspect
from IPython.display import display, Markdown

MAX_NEW_TOKENS = 2048   # 1 ターンあたりの最大生成トークン数
MAX_TURNS = 6           # Tool 呼び出しの最大往復数(無限ループ防止)

SYSTEM_PROMPT = """あなたは日本の天気について答えるアシスタントです。今日は {today} です。

ルール:
- 天気・気温・降水確率・警報・観測値などの情報は、必ず提供された関数(Tool)で気象庁のデータを取得してから答えてください。自分の記憶や推測で答えてはいけません。
- 複数の情報が必要なときは、関数を順番に呼び出してください。数値をまとめて集計するときは calc_stats を使ってください。
- 関数から取得した値は、そのまま正確に使ってください(丸めたり、勝手に補ったりしない)。
- 最終回答は日本語の Markdown で書いてください。日別・地点別のデータは表にまとめ、見出しを付け、最後に「出典: 気象庁(発表 日時)」を添えてください。
"""

TOOL_CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)


def build_system_prompt() -> str:
    today = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9)))
    weekdays = ["月", "火", "水", "木", "金", "土", "日"]
    return SYSTEM_PROMPT.format(today=today.strftime(f"%Y年%m月%d日({weekdays[today.weekday()]}) %H:%M"))


def run_llm(messages, tools):
    """メッセージ列(と Tool 定義)を LLM に渡し、生成テキストを返す"""
    text = tokenizer.apply_chat_template(
        messages,
        tools=tools if tools else None,
        add_generation_prompt=True,
        tokenize=False,
        enable_thinking=False,   # 思考モード OFF(速度優先)
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def parse_tool_calls(text: str):
    """LLM の出力から <tool_call>{...}</tool_call> をすべて取り出す"""
    calls = []
    for m in TOOL_CALL_RE.finditer(text):
        try:
            obj = json.loads(m.group(1))
            if "name" in obj:
                calls.append({"name": obj["name"], "arguments": obj.get("arguments", {}) or {}})
        except json.JSONDecodeError:
            calls.append({"name": "__parse_error__", "arguments": {"raw": m.group(1)}})
    return calls


def execute_tool(call, registry):
    """Tool を実行して結果(dict)を返す。失敗しても例外を LLM に返して続行させる"""
    name, args = call["name"], call["arguments"]
    if name not in registry:
        return {"エラー": f"関数 {name} は登録されていません。使える関数: {list(registry)}"}
    try:
        if not isinstance(args, dict):
            # 引数がオブジェクトでない場合は最初の引数に割り当てる
            first = list(inspect.signature(registry[name]).parameters)[0]
            args = {first: args}
        return registry[name](**args)
    except Exception as e:
        return {"エラー": f"{type(e).__name__}: {e}"}


def _fmt_args(args) -> str:
    return ", ".join(f"{k}={json.dumps(v, ensure_ascii=False)}" for k, v in args.items())


def ask(question: str, tools=None, verbose: bool = True, show: bool = True, max_turns: int = MAX_TURNS) -> str:
    """
    質問文を LLM に渡し、必要に応じて Tool を呼び出しながら最終回答(Markdown)を返す。

    tools   : LLM に使わせる関数のリスト。None なら LLM 単体で回答させる
    verbose : Tool 呼び出しの過程をターンごとに表示する
    show    : 最終回答を Markdown として描画する
    """
    registry = {f.__name__: f for f in (tools or [])}
    messages = [
        {"role": "system", "content": build_system_prompt()},
        {"role": "user", "content": question},
    ]
    if verbose:
        print(f"❓ 質問: {question}")
        print(f"🧰 使える Tool: {list(registry) if registry else 'なし(LLM 単体)'}\n")

    output = ""
    for turn in range(1, max_turns + 1):
        output = run_llm(messages, tools)
        calls = parse_tool_calls(output) if registry else []

        if not calls:
            if verbose:
                print(f"✅ [ターン {turn}] 最終回答(Tool 呼び出しなし)\n")
            break

        # --- LLM が Tool を呼ぶと判断した ---
        content = TOOL_CALL_RE.sub("", output).strip()
        messages.append({
            "role": "assistant",
            "content": content,
            "tool_calls": [{"type": "function", "function": c} for c in calls],
        })
        for c in calls:
            if verbose:
                print(f"🔵 [ターン {turn}] LLM の判断 → Tool 呼び出し: {c['name']}({_fmt_args(c['arguments'])})")
            result = execute_tool(c, registry)
            result_json = json.dumps(result, ensure_ascii=False)
            if verbose:
                head = result_json[:120].replace("\n", " ")
                print(f"🟢 [Tool 応答] {c['name']} → {len(result_json):,} 文字  {head}…\n")
            messages.append({"role": "tool", "name": c["name"], "content": result_json})
    else:
        if verbose:
            print(f"⚠️ {max_turns} ターンに達したため打ち切りました")

    if show:
        display(Markdown(output))
    return output

## 3. まず、LLM 単体に聞いてみる

Tool を渡さずに、明日の天気を聞いてみます。LLM は学習した時点までの知識しか持っておらず、
今日の日付も、今の気圧配置も知りません。それでも **もっともらしく答えてしまう** のを見てください。

In [ ]:
question = "明日の京都の天気と最高気温を教えて" #@param {type:"string"}
answer = ask(question, tools=None)

上の回答は、一見それらしいですが **根拠がありません**(季節の一般論か、学習データにあった過去の天気の記憶です)。
これが、LLM を天気に使うときの出発点の問題です。

## 4. Tool を 1 つ渡す — 週間予報

気象庁の週間予報を取ってくる関数 `get_weekly_forecast` を書いて、LLM に渡します。
関数の中身は「気象庁の JSON を取得して、読みやすい辞書に整える」だけです。
LLM に読ませるのは **docstring(関数の説明文)** なので、そこに「何ができるか」「引数は何か」を日本語で丁寧に書くのがコツです。

まず共通部分(地域名→コード表、天気コード表)を用意します。

In [ ]:
#@title 共通部分 — 気象庁 JSON の取得、地域テーブル、天気コード表
import json
import datetime
import urllib.request
from typing import List, Optional

JMA_BASE = "https://www.jma.go.jp/bosai"
_CACHE = {}


def _fetch_json(path: str, ttl_sec: int = 300):
    """気象庁の JSON を取得する(同じ URL は ttl_sec 秒キャッシュ)"""
    now = datetime.datetime.now().timestamp()
    if path in _CACHE and now - _CACHE[path][0] < ttl_sec:
        return _CACHE[path][1]
    req = urllib.request.Request(f"{JMA_BASE}/{path}", headers={"User-Agent": "jma-tool-use-weather/1.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        data = json.loads(r.read().decode("utf-8"))
    _CACHE[path] = (now, data)
    return data


def _fetch_text(path: str) -> str:
    req = urllib.request.Request(f"{JMA_BASE}/{path}", headers={"User-Agent": "jma-tool-use-weather/1.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return r.read().decode("utf-8").strip()


# ---------- 地域テーブル(予報の発表単位 = 気象台の担当区域) ----------
# 値: (予報区コード, 正式名称, アメダス府県番号のリスト)
AREA_TABLE = {
    "宗谷地方":        ("011000", "宗谷地方", ["11"]),
    "上川・留萌地方":  ("012000", "上川・留萌地方", ["12", "13"]),
    "網走・北見・紋別地方": ("013000", "網走・北見・紋別地方", ["17"]),
    "十勝地方":        ("014030", "十勝地方", ["20"]),
    "釧路・根室地方":  ("014100", "釧路・根室地方", ["18", "19"]),
    "胆振・日高地方":  ("015000", "胆振・日高地方", ["21", "22"]),
    "石狩・空知・後志地方": ("016000", "石狩・空知・後志地方", ["14", "15", "16"]),
    "渡島・檜山地方":  ("017000", "渡島・檜山地方", ["23", "24"]),
    "青森県": ("020000", "青森県", ["31"]), "岩手県": ("030000", "岩手県", ["33"]),
    "宮城県": ("040000", "宮城県", ["34"]), "秋田県": ("050000", "秋田県", ["32"]),
    "山形県": ("060000", "山形県", ["35"]), "福島県": ("070000", "福島県", ["36"]),
    "茨城県": ("080000", "茨城県", ["40"]), "栃木県": ("090000", "栃木県", ["41"]),
    "群馬県": ("100000", "群馬県", ["42"]), "埼玉県": ("110000", "埼玉県", ["43"]),
    "千葉県": ("120000", "千葉県", ["45"]), "東京都": ("130000", "東京都", ["44"]),
    "神奈川県": ("140000", "神奈川県", ["46"]), "新潟県": ("150000", "新潟県", ["54"]),
    "富山県": ("160000", "富山県", ["55"]), "石川県": ("170000", "石川県", ["56"]),
    "福井県": ("180000", "福井県", ["57"]), "山梨県": ("190000", "山梨県", ["49"]),
    "長野県": ("200000", "長野県", ["48"]), "岐阜県": ("210000", "岐阜県", ["52"]),
    "静岡県": ("220000", "静岡県", ["50"]), "愛知県": ("230000", "愛知県", ["51"]),
    "三重県": ("240000", "三重県", ["53"]), "滋賀県": ("250000", "滋賀県", ["60"]),
    "京都府": ("260000", "京都府", ["61"]), "大阪府": ("270000", "大阪府", ["62"]),
    "兵庫県": ("280000", "兵庫県", ["63"]), "奈良県": ("290000", "奈良県", ["64"]),
    "和歌山県": ("300000", "和歌山県", ["65"]), "鳥取県": ("310000", "鳥取県", ["69"]),
    "島根県": ("320000", "島根県", ["68"]), "岡山県": ("330000", "岡山県", ["66"]),
    "広島県": ("340000", "広島県", ["67"]), "山口県": ("350000", "山口県", ["81"]),
    "徳島県": ("360000", "徳島県", ["72"]), "香川県": ("370000", "香川県", ["71"]),
    "愛媛県": ("380000", "愛媛県", ["73"]), "高知県": ("390000", "高知県", ["74"]),
    "福岡県": ("400000", "福岡県", ["82"]), "佐賀県": ("410000", "佐賀県", ["85"]),
    "長崎県": ("420000", "長崎県", ["84"]), "熊本県": ("430000", "熊本県", ["86"]),
    "大分県": ("440000", "大分県", ["83"]), "宮崎県": ("450000", "宮崎県", ["87"]),
    "鹿児島県": ("460100", "鹿児島県(奄美地方を除く)", ["88"]),
    "奄美地方": ("460040", "奄美地方", ["88"]),
    "沖縄本島地方": ("471000", "沖縄本島地方", ["91"]),
    "大東島地方": ("472000", "大東島地方", ["92"]),
    "宮古島地方": ("473000", "宮古島地方", ["93"]),
    "八重山地方": ("474000", "八重山地方", ["94"]),
}
# 略称・別名 → 正式キー
AREA_ALIASES = {
    "北海道": "石狩・空知・後志地方", "札幌": "石狩・空知・後志地方", "石狩": "石狩・空知・後志地方",
    "旭川": "上川・留萌地方", "上川": "上川・留萌地方", "稚内": "宗谷地方", "宗谷": "宗谷地方",
    "網走": "網走・北見・紋別地方", "北見": "網走・北見・紋別地方", "オホーツク": "網走・北見・紋別地方",
    "帯広": "十勝地方", "十勝": "十勝地方", "釧路": "釧路・根室地方", "根室": "釧路・根室地方",
    "室蘭": "胆振・日高地方", "苫小牧": "胆振・日高地方", "函館": "渡島・檜山地方",
    "東京": "東京都", "京都": "京都府", "大阪": "大阪府", "沖縄": "沖縄本島地方", "沖縄県": "沖縄本島地方",
    "那覇": "沖縄本島地方", "奄美": "奄美地方", "宮古島": "宮古島地方", "宮古": "宮古島地方",
    "石垣": "八重山地方", "石垣島": "八重山地方", "八重山": "八重山地方", "大東島": "大東島地方",
}
for _k in list(AREA_TABLE):
    if _k.endswith("県"):
        AREA_ALIASES[_k[:-1]] = _k


def resolve_area(area: str):
    """入力された地域名を (予報区コード, 正式名称, アメダス府県番号) に解決する"""
    a = area.strip().replace(" ", "").replace("　", "")
    if a in AREA_TABLE:
        return AREA_TABLE[a]
    if a in AREA_ALIASES:
        return AREA_TABLE[AREA_ALIASES[a]]
    # 前方一致(例: 「京都市」→ 京都府)
    for k in list(AREA_ALIASES) + list(AREA_TABLE):
        if a.startswith(k):
            return AREA_TABLE[AREA_ALIASES.get(k, k)]
    raise ValueError(
        f"地域「{area}」を解決できません。都道府県名(例: 京都府、東京都)または "
        f"北海道・沖縄の地方名(例: 石狩・空知・後志地方、沖縄本島地方)で指定してください。"
    )


WEEKDAYS = ["月", "火", "水", "木", "金", "土", "日"]


def _date_label(iso: str) -> str:
    d = datetime.date.fromisoformat(iso[:10])
    return f"{d.month}/{d.day}({WEEKDAYS[d.weekday()]})"

# ---------- 天気コード表(気象庁 TELOPS、118 種。気象庁ページの定数表から作成) ----------
WEATHER_CODES = {
"100": "晴",
"101": "晴時々曇",
"102": "晴一時雨",
"103": "晴時々雨",
"104": "晴一時雪",
"105": "晴時々雪",
"106": "晴一時雨か雪",
"107": "晴時々雨か雪",
"108": "晴一時雨か雷雨",
"110": "晴後時々曇",
"111": "晴後曇",
"112": "晴後一時雨",
"113": "晴後時々雨",
"114": "晴後雨",
"115": "晴後一時雪",
"116": "晴後時々雪",
"117": "晴後雪",
"118": "晴後雨か雪",
"119": "晴後雨か雷雨",
"120": "晴朝夕一時雨",
"121": "晴朝の内一時雨",
"122": "晴夕方一時雨",
"123": "晴山沿い雷雨",
"124": "晴山沿い雪",
"125": "晴午後は雷雨",
"126": "晴昼頃から雨",
"127": "晴夕方から雨",
"128": "晴夜は雨",
"130": "朝の内霧後晴",
"131": "晴明け方霧",
"132": "晴朝夕曇",
"140": "晴時々雨で雷を伴う",
"160": "晴一時雪か雨",
"170": "晴時々雪か雨",
"181": "晴後雪か雨",
"200": "曇",
"201": "曇時々晴",
"202": "曇一時雨",
"203": "曇時々雨",
"204": "曇一時雪",
"205": "曇時々雪",
"206": "曇一時雨か雪",
"207": "曇時々雨か雪",
"208": "曇一時雨か雷雨",
"209": "霧",
"210": "曇後時々晴",
"211": "曇後晴",
"212": "曇後一時雨",
"213": "曇後時々雨",
"214": "曇後雨",
"215": "曇後一時雪",
"216": "曇後時々雪",
"217": "曇後雪",
"218": "曇後雨か雪",
"219": "曇後雨か雷雨",
"220": "曇朝夕一時雨",
"221": "曇朝の内一時雨",
"222": "曇夕方一時雨",
"223": "曇日中時々晴",
"224": "曇昼頃から雨",
"225": "曇夕方から雨",
"226": "曇夜は雨",
"228": "曇昼頃から雪",
"229": "曇夕方から雪",
"230": "曇夜は雪",
"231": "曇海上海岸は霧か霧雨",
"240": "曇時々雨で雷を伴う",
"250": "曇時々雪で雷を伴う",
"260": "曇一時雪か雨",
"270": "曇時々雪か雨",
"281": "曇後雪か雨",
"300": "雨",
"301": "雨時々晴",
"302": "雨時々止む",
"303": "雨時々雪",
"304": "雨か雪",
"306": "大雨",
"308": "雨で暴風を伴う",
"309": "雨一時雪",
"311": "雨後晴",
"313": "雨後曇",
"314": "雨後時々雪",
"315": "雨後雪",
"316": "雨か雪後晴",
"317": "雨か雪後曇",
"320": "朝の内雨後晴",
"321": "朝の内雨後曇",
"322": "雨朝晩一時雪",
"323": "雨昼頃から晴",
"324": "雨夕方から晴",
"325": "雨夜は晴",
"326": "雨夕方から雪",
"327": "雨夜は雪",
"328": "雨一時強く降る",
"329": "雨一時みぞれ",
"340": "雪か雨",
"350": "雨で雷を伴う",
"361": "雪か雨後晴",
"371": "雪か雨後曇",
"400": "雪",
"401": "雪時々晴",
"402": "雪時々止む",
"403": "雪時々雨",
"405": "大雪",
"406": "風雪強い",
"407": "暴風雪",
"409": "雪一時雨",
"411": "雪後晴",
"413": "雪後曇",
"414": "雪後雨",
"420": "朝の内雪後晴",
"421": "朝の内雪後曇",
"422": "雪昼頃から雨",
"423": "雪夕方から雨",
"425": "雪一時強く降る",
"426": "雪後みぞれ",
"427": "雪一時みぞれ",
"450": "雪で雷を伴う"
}

In [ ]:
#@title Tool 1: 週間予報
# ============================================================
# Tool 1: 週間予報
# ============================================================
def get_weekly_forecast(area: str) -> dict:
    """
    指定した都道府県の週間天気予報(明日から 7 日分)を気象庁から取得します。
    日別の天気、降水確率、最低気温・最高気温(予測の幅つき)、予報の信頼度(A/B/C)を返します。

    Args:
        area: 都道府県名(例: "京都府", "東京都", "北海道", "沖縄県")。「京都」のような略称も可。
    """
    code, name, _ = resolve_area(area)
    data = _fetch_json(f"forecast/data/forecast/{code}.json")
    if len(data) < 2:
        return {"地域": name, "エラー": "週間予報が提供されていません"}
    wk = data[1]
    ts_w = wk["timeSeries"][0]          # 天気・降水確率・信頼度
    ts_t = wk["timeSeries"][1]          # 気温(代表地点)
    wa = ts_w["areas"][0]
    ta = ts_t["areas"][0]
    days = []
    for i, t in enumerate(ts_w["timeDefines"]):
        def g(lst, idx):
            return lst[idx] if idx < len(lst) and lst[idx] != "" else None
        days.append({
            "日付": _date_label(t),
            "天気": WEATHER_CODES.get(wa["weatherCodes"][i], f"コード{wa['weatherCodes'][i]}"),
            "降水確率(%)": g(wa["pops"], i),
            "最低気温(℃)": g(ta["tempsMin"], i),
            "最低気温の幅": (f"{g(ta['tempsMinLower'], i)}〜{g(ta['tempsMinUpper'], i)}"
                          if g(ta["tempsMinLower"], i) else None),
            "最高気温(℃)": g(ta["tempsMax"], i),
            "最高気温の幅": (f"{g(ta['tempsMaxLower'], i)}〜{g(ta['tempsMaxUpper'], i)}"
                          if g(ta["tempsMaxLower"], i) else None),
            "信頼度": g(wa["reliabilities"], i),
        })
    # 週間予報の初日(明日)の気温は 3 日間予報側にあるので補完する
    try:
        ts_s = data[0]["timeSeries"][2]
        sa = next((x for x in ts_s["areas"] if x["area"]["name"] == ta["area"]["name"]), ts_s["areas"][0])
        by_date = {}
        for i, t in enumerate(ts_s["timeDefines"]):
            by_date.setdefault(_date_label(t), []).append(sa["temps"][i])
        for d in days:
            if d["最低気温(℃)"] is None and d["日付"] in by_date and len(by_date[d["日付"]]) >= 2:
                d["最低気温(℃)"], d["最高気温(℃)"] = by_date[d["日付"]][0], by_date[d["日付"]][1]
                d["補足"] = "気温は 3 日間予報の値。降水確率は 3 日間予報(get_short_forecast)を参照"
    except Exception:
        pass

    result = {
        "地域": name,
        "予報区": wa["area"]["name"],
        "気温の代表地点": ta["area"]["name"],
        "発表": wk["reportDatetime"][:16].replace("T", " "),
        "発表機関": wk["publishingOffice"],
        "日別予報": days,
        "備考": "信頼度 A=確度が高い, B=やや高い, C=低い。気温の幅は予測の振れ幅。初日は当日分のため一部空欄。",
    }
    if "tempAverage" in wk:
        av = wk["tempAverage"]["areas"][0]
        result["この時期の平年値"] = {"最低気温(℃)": av["min"], "最高気温(℃)": av["max"]}
    return result


# --- LLM 抜きで、関数を直接呼んでみる(Tool は「ただの関数」です) ---
import json
print(json.dumps(get_weekly_forecast("京都"), ensure_ascii=False, indent=1)[:1500], "...")

関数が動くことを確認できたので、**同じ質問**を、今度はこの関数を渡して聞いてみます。
`🔵 LLM の判断 → Tool 呼び出し` の行に注目してください。LLM が「この質問には週間予報が要る」と判断して、
引数に「京都府」を入れて関数を呼んでいます。

In [ ]:
question = "明日の京都の天気と最高気温を教えて" #@param {type:"string"}
answer = ask(question, tools=[get_weekly_forecast])

## 5. Tool を増やす

残りの Tool を登録します。

| Tool | 取ってくるもの |
|---|---|
| `get_short_forecast` | 今日・明日・明後日の天気・風・6 時間ごとの降水確率・気温 |
| `get_forecast_overview` | 気象台の予報官が書いた概況文 |
| `get_warnings` | 発表中の警報・注意報・危険警報(市町村名つき) |
| `get_amedas_now` | 府県内のアメダス地点の最新実況(気温・降水・風・湿度・積雪) |
| `calc_stats` | 数値リストの平均・最大・最小・合計 |

> 参考: 警報・注意報は **2026 年 5 月 28 日に体系が変わり**(「危険警報」の新設、洪水警報の河川ごとの氾濫情報への分離)、種類コードの意味も変わりました。
> この関数の対応表は、気象庁の警報ページが実際に使っている定義から作っています。古い資料や AI の記憶にある対応表は当てになりません。

In [ ]:
#@title Tool 2〜6
# ---------- 警報・注意報の種類コード(2026-05-28 改正後の体系) ----------
# 気象庁の警報ページが実際に使っている対応表から作成。レベル: 注意報 < 警報 < 危険警報 < 特別警報
WARNING_CODES = {
    "10": "大雨注意報", "03": "大雨警報", "43": "大雨危険警報", "33": "大雨特別警報",
    "29": "土砂災害注意報", "09": "土砂災害警報", "49": "土砂災害危険警報", "39": "土砂災害特別警報",
    "19": "高潮注意報", "08": "高潮警報", "48": "高潮危険警報", "38": "高潮特別警報",
    "15": "強風注意報", "05": "暴風警報", "35": "暴風特別警報",
    "13": "風雪注意報", "02": "暴風雪警報", "32": "暴風雪特別警報",
    "12": "大雪注意報", "06": "大雪警報", "36": "大雪特別警報",
    "16": "波浪注意報", "07": "波浪警報", "37": "波浪特別警報",
    "14": "雷注意報", "17": "融雪注意報", "20": "濃霧注意報", "21": "乾燥注意報",
    "22": "なだれ注意報", "23": "低温注意報", "24": "霜注意報", "25": "着氷注意報", "26": "着雪注意報",
}
WARNING_LEVEL = {"注意報": 2, "警報": 3, "危険警報": 4, "特別警報": 5}


def _warning_level(name: str) -> int:
    for suffix, lv in sorted(WARNING_LEVEL.items(), key=lambda x: -len(x[0])):
        if name.endswith(suffix):
            return lv
    return 0

# ============================================================
# Tool 2: 3 日間予報(今日・明日・明後日)
# ============================================================
def get_short_forecast(area: str) -> dict:
    """
    指定した都道府県の今日・明日・明後日の天気予報を気象庁から取得します。
    地域(南部・北部など)ごとの天気と風、6 時間ごとの降水確率、今日・明日の最低気温・最高気温を返します。

    Args:
        area: 都道府県名(例: "京都府", "東京都", "北海道", "沖縄県")。「京都」のような略称も可。
    """
    code, name, _ = resolve_area(area)
    data = _fetch_json(f"forecast/data/forecast/{code}.json")
    sh = data[0]
    ts_w, ts_p, ts_t = sh["timeSeries"][0], sh["timeSeries"][1], sh["timeSeries"][2]

    regions = []
    for ar in ts_w["areas"]:
        rows = []
        for i, t in enumerate(ts_w["timeDefines"]):
            row = {"日付": _date_label(t),
                   "天気": ar["weathers"][i].replace("　", " "),
                   "風": ar["winds"][i].replace("　", " ")}
            if "waves" in ar:
                row["波"] = ar["waves"][i].replace("　", " ")
            rows.append(row)
        regions.append({"地域": ar["area"]["name"], "予報": rows})

    pops = []
    for ar in ts_p["areas"]:
        seq = []
        for i, t in enumerate(ts_p["timeDefines"]):
            hh = int(t[11:13])
            seq.append({"時間帯": f"{_date_label(t)} {hh:02d}時〜{(hh + 6) % 24:02d}時",
                        "降水確率(%)": ar["pops"][i]})
        pops.append({"地域": ar["area"]["name"], "降水確率": seq})

    temps = []
    for ar in ts_t["areas"]:
        by_date = {}
        for i, t in enumerate(ts_t["timeDefines"]):
            by_date.setdefault(_date_label(t), []).append(ar["temps"][i])
        rows = []
        for d, vals in by_date.items():
            if len(vals) >= 2:
                rows.append({"日付": d, "最低気温(℃)": vals[0], "最高気温(℃)": vals[1]})
            else:
                rows.append({"日付": d, "気温(℃)": vals[0]})
        temps.append({"地点": ar["area"]["name"], "気温": rows})

    return {
        "地域": name,
        "発表": sh["reportDatetime"][:16].replace("T", " "),
        "発表機関": sh["publishingOffice"],
        "天気と風": regions,
        "降水確率": pops,
        "気温": temps,
    }


# ============================================================
# Tool 3: 概況文(予報官による解説)
# ============================================================
def get_forecast_overview(area: str) -> dict:
    """
    指定した都道府県の天気概況(気象台の予報官が書いた日本語の解説文)を気象庁から取得します。
    今日・明日の天気の背景(前線、高気圧、湿った空気など)を知りたいとき、
    予報を文章として要約・記事化したいときに使います。

    Args:
        area: 都道府県名(例: "京都府", "東京都", "北海道", "沖縄県")。「京都」のような略称も可。
    """
    code, name, _ = resolve_area(area)
    d = _fetch_json(f"forecast/data/overview_forecast/{code}.json")
    return {
        "地域": name,
        "発表": d["reportDatetime"][:16].replace("T", " "),
        "発表機関": d["publishingOffice"],
        "見出し": d.get("headlineText", ""),
        "概況文": d["text"].replace("　", "").strip(),
    }


# ============================================================
# Tool 4: 警報・注意報
# ============================================================
def get_warnings(area: str) -> dict:
    """
    指定した都道府県で現在発表中の気象警報・注意報・危険警報・特別警報を気象庁から取得します。
    種類ごとに、発表されている市町村名の一覧を返します。発表中のものがなければその旨を返します。

    Args:
        area: 都道府県名(例: "京都府", "東京都", "北海道", "沖縄県")。「京都」のような略称も可。
    """
    code, name, _ = resolve_area(area)
    entries = _fetch_json(f"warning/data/r8/{code}.json", ttl_sec=120)
    area_tbl = _fetch_json("common/const/area.json", ttl_sec=86400)
    names = {}
    for lv in ("class10s", "class15s", "class20s"):
        for k, v in area_tbl[lv].items():
            names[k] = v["name"]

    inactive = ("解除", "発表警報・注意報はなし")
    by_type = {}       # 種類名 -> set(市町村)
    headlines = []
    latest = ""
    for e in entries:                       # 情報種別(大雨・土砂災害 / 風・雪 / 波浪・高潮 / 雷・霧 など)ごと
        w = e.get("warning", {})
        items = w.get("class20Items") or w.get("class10Items") or []
        active_here = False
        for it in items:
            for kd in it.get("kinds", []):
                if not kd.get("code") or kd.get("status") in inactive:
                    continue
                label = WARNING_CODES.get(kd["code"], f"種類コード{kd['code']}")
                if kd.get("additions"):
                    label += "(" + "・".join(kd["additions"]) + ")"
                by_type.setdefault(label, set()).add(names.get(it["areaCode"], it["areaCode"]))
                active_here = True
        if active_here:
            if e.get("headlineText"):
                headlines.append(e["headlineText"])
            latest = max(latest, e.get("reportDatetime", ""))

    result = {
        "地域": name,
        "発表機関": entries[0]["publishingOffice"] if entries else "",
    }
    if by_type:
        rows = []
        for k, v in by_type.items():
            munis = sorted(v)
            row = {"種類": k, "市町村数": len(munis)}
            if len(munis) > 20:
                row["対象市町村"] = munis[:20] + [f"ほか{len(munis) - 20}市町村"]
            else:
                row["対象市町村"] = munis
            rows.append(row)
        rows.sort(key=lambda r: -_warning_level(r["種類"].split("(")[0]))
        result["発表"] = latest[:16].replace("T", " ")
        result["見出し"] = headlines
        result["発表中の警報・注意報"] = rows
    else:
        result["発表中の警報・注意報"] = "現在、発表中の警報・注意報はありません"
    result["備考"] = "レベルは 注意報 < 警報 < 危険警報 < 特別警報 の順に高い。防災上の判断は気象庁・自治体の公式発表に基づくこと。"
    return result


# ============================================================
# Tool 5: アメダス実況(最新 10 分値)
# ============================================================
WIND_DIRS = ["静穏", "北北東", "北東", "東北東", "東", "東南東", "南東", "南南東", "南",
             "南南西", "南西", "西南西", "西", "西北西", "北西", "北北西", "北"]


def get_amedas_now(area: str, max_stations: int = 30) -> dict:
    """
    指定した都道府県内のアメダス観測所の最新の実況値(気温、1 時間降水量、風向・風速、湿度、積雪深)を
    気象庁から取得します。「今の気温は?」「今雨が降っている所は?」のような現在の状況を知りたいときに使います。

    Args:
        area: 都道府県名(例: "京都府", "東京都", "北海道", "沖縄県")。「京都」のような略称も可。
        max_stations: 返す観測所の最大数(既定 30)。
    """
    _, name, prefs = resolve_area(area)
    latest = _fetch_text("amedas/data/latest_time.txt")
    key = datetime.datetime.fromisoformat(latest).strftime("%Y%m%d%H%M%S")
    obs = _fetch_json(f"amedas/data/map/{key}.json", ttl_sec=600)
    tbl = _fetch_json("amedas/const/amedastable.json", ttl_sec=86400)

    def val(rec, k):
        v = rec.get(k)
        return v[0] if v and v[1] == 0 and v[0] is not None else None

    rows = []
    for sid, meta in tbl.items():
        if sid[:2] not in prefs or sid not in obs:
            continue
        rec = obs[sid]
        if val(rec, "temp") is None and val(rec, "precipitation1h") is None:
            continue
        wd = val(rec, "windDirection")
        rows.append({
            "観測所": meta["kjName"],
            "標高(m)": meta.get("alt"),
            "気温(℃)": val(rec, "temp"),
            "1時間降水量(mm)": val(rec, "precipitation1h"),
            "風向": WIND_DIRS[wd] if wd is not None and 0 <= wd < len(WIND_DIRS) else None,
            "風速(m/s)": val(rec, "wind"),
            "湿度(%)": val(rec, "humidity"),
            "積雪深(cm)": val(rec, "snow"),
        })
    # 気温がある地点を優先し、標高の低い順(平地の代表地点が先に来る)
    rows.sort(key=lambda r: (r["気温(℃)"] is None, r["標高(m)"] or 0))
    rows = rows[:max_stations]
    temps = [r["気温(℃)"] for r in rows if r["気温(℃)"] is not None]
    summary = {}
    if temps:
        hi = max(rows, key=lambda r: r["気温(℃)"] if r["気温(℃)"] is not None else -999)
        lo = min(rows, key=lambda r: r["気温(℃)"] if r["気温(℃)"] is not None else 999)
        summary = {"最高": f"{hi['気温(℃)']}℃({hi['観測所']})", "最低": f"{lo['気温(℃)']}℃({lo['観測所']})"}
    return {
        "地域": name,
        "観測時刻": latest[:16].replace("T", " "),
        "地点数": len(rows),
        "気温の範囲": summary,
        "観測値": rows,
    }


# ============================================================
# Tool 6: 統計計算
# ============================================================
def calc_stats(values: List[float]) -> dict:
    """
    数値のリストの平均・最大・最小・合計・個数を計算します。
    気温や降水確率など、他の Tool で得た数値をまとめて集計したいときに使います。

    Args:
        values: 数値のリスト(例: [24, 25, 27, 26])。文字列の数値も可。
    """
    nums = []
    for v in values:
        if v is None or v == "":
            continue
        nums.append(float(str(v).replace("℃", "").replace("%", "")))
    if not nums:
        return {"エラー": "数値がありません"}
    return {
        "個数": len(nums),
        "平均": round(sum(nums) / len(nums), 1),
        "最大": max(nums),
        "最小": min(nums),
        "合計": round(sum(nums), 1),
    }

TOOLS = [get_weekly_forecast, get_short_forecast, get_forecast_overview,
         get_warnings, get_amedas_now, calc_stats]

print("登録した Tool:", [f.__name__ for f in TOOLS])

In [ ]:
#@title Tool を直接呼んで動作確認(LLM 抜き)
import json
print(json.dumps(get_warnings("京都"), ensure_ascii=False, indent=1)[:800], "...\n")
print(json.dumps(get_amedas_now("京都", max_stations=5), ensure_ascii=False, indent=1)[:900], "...")

### 2 段階の Tool 呼び出し

「今週の最高気温の平均」は、週間予報を取ってから、その数値を集計する必要があります。
LLM が **自分で 2 つの Tool を順番に呼ぶ** 様子を見てください。

In [ ]:
question = "京都の今週の最高気温の平均は?" #@param {type:"string"}
answer = ask(question, tools=TOOLS)

## 6. 質問例いろいろ

`question` を書き換えて試してください。分野別の例:

- **生活**: 「今週の京都でサイクリングに向く日は?」「北海道と沖縄、今週どちらが寒暖差が大きい?」
- **風力**: 「今日から明後日まで、青森県で風が強そうな日と時間帯は?」
- **寒地**: 「北海道で今週いちばん冷え込む日は?」
- **山岳**: 「長野県の週間予報で、信頼度が低い日はどれ?」
- **防災**: 「今、近畿地方で警報・注意報が出ている府県と市町村は?」(複数の府県を順に呼ぶ)
- **気象データアナリスト**: 「今の京都の気温は、今日の予報の最高気温とどれくらい違う?」(実況と予報の照合)
- **解説**: 「今日の京都の天気の背景を、気象台の概況文をもとに説明して」

In [ ]:
question = "今週の京都でサイクリングに向く日は? 理由も" #@param {type:"string"}
answer = ask(question, tools=TOOLS)

In [ ]:
question = "今の京都の気温は、今日の予報の最高気温とどれくらい違う?" #@param {type:"string"}
answer = ask(question, tools=TOOLS)

In [ ]:
question = "今、京都府・大阪府・兵庫県で警報・注意報が出ている市町村は?" #@param {type:"string"}
answer = ask(question, tools=TOOLS)

## 7. 記事・お知らせ文を作って保存する

LLM の本領は「数字を、用途に合った文章に変える」ことです。同じ Tool で、記事、掲示文、子ども向け説明が作れます。
回答は Markdown(見出し・表)で表示され、`save_answer` で `.md` と `.html` に保存してダウンロードできます。

In [ ]:
#@title 保存・ダウンロード用の関数
import os
import re as _re


def save_answer(markdown_text: str, title: str = "answer", download: bool = True):
    """回答(Markdown)を output/ に .md と .html で保存し、Colab ならダウンロードする"""
    os.makedirs("output", exist_ok=True)
    stamp = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime("%Y%m%d_%H%M")
    slug = _re.sub(r"[^\w\-]+", "_", title)[:40]
    base = f"output/{stamp}_{slug}"
    with open(base + ".md", "w", encoding="utf-8") as f:
        f.write(markdown_text)
    try:
        import markdown as _md
        body = _md.markdown(markdown_text, extensions=["tables", "fenced_code"])
    except ImportError:
        body = "<pre>" + markdown_text + "</pre>"
    html = f"""<!doctype html><html lang="ja"><head><meta charset="utf-8"><title>{title}</title>
<style>body{{font-family:-apple-system,'Hiragino Sans','Noto Sans JP',sans-serif;max-width:800px;margin:2em auto;padding:0 1em;line-height:1.7}}
table{{border-collapse:collapse;margin:1em 0}}th,td{{border:1px solid #ccc;padding:4px 10px}}th{{background:#f3f3f3}}</style></head>
<body>{body}</body></html>"""
    with open(base + ".html", "w", encoding="utf-8") as f:
        f.write(html)
    print(f"💾 保存しました: {base}.md / {base}.html")
    if download:
        try:
            from google.colab import files
            files.download(base + ".md")
            files.download(base + ".html")
        except ImportError:
            pass
    return base

In [ ]:
question = "宮城県の今週の天気を、地方紙の天気コーナー向けに 300 字程度の記事にして。見出しと、日別の表も付けて" #@param {type:"string"}
answer = ask(question, tools=TOOLS)
save_answer(answer, title="宮城県_週間天気記事")

In [ ]:
question = "京都府の今日と明日の天気を、小学生にも分かる言葉で、注意することも含めて説明して" #@param {type:"string"}
answer = ask(question, tools=TOOLS)
save_answer(answer, title="京都_子ども向け")

## 8. 落とし穴 — LLM は Tool の間違いを隠す

ここが、気象データを扱う人にとっていちばん大事な実験です。
週間予報の関数を **わざと壊して**(最高気温に +10℃ する)、同じ質問をしてみます。
LLM は関数の出力を疑わずに、自信たっぷりに答えます。

In [ ]:
#@title 週間予報の Tool をわざと壊す
_get_weekly_forecast_ok = get_weekly_forecast   # 正しい版を退避

def get_weekly_forecast(area: str) -> dict:
    """
    指定した都道府県の週間天気予報(明日から 7 日分)を気象庁から取得します。

    Args:
        area: 都道府県名(例: "京都府")
    """
    r = _get_weekly_forecast_ok(area)
    for d in r["日別予報"]:
        if d.get("最高気温(℃)"):
            d["最高気温(℃)"] = str(int(d["最高気温(℃)"]) + 10)   # ← バグ(のつもり)
    return r

BROKEN_TOOLS = [get_weekly_forecast] + [f for f in TOOLS if f.__name__ != "get_weekly_forecast"]
answer = ask("京都の今週の最高気温を日別に教えて。今の季節として普通?", tools=BROKEN_TOOLS)

In [ ]:
#@title 正しい版に戻す
get_weekly_forecast = _get_weekly_forecast_ok
TOOLS = [get_weekly_forecast, get_short_forecast, get_forecast_overview, get_warnings, get_amedas_now, calc_stats]
print("✅ 元に戻しました")

### ここから言えること

- LLM は **渡されたデータを信じる**。10 月の京都で最高気温 35℃ でも「暑い日が続きます」と書く
- だから、この仕組みで検証すべきは LLM の文章力ではなく、**Tool と、その元データ**
- 逆に言えば、Tool を気象庁の公式データに固定しているからこそ、この窓口は信頼できる
- 実際にこのノートブックを作る過程でも、警報コードの対応表を AI の記憶で書いていたら **2026 年 5 月の改正前の古い意味** で名称を返すところでした。気象庁のページから現行の表を取ってきて防いでいます

もう一つの落とし穴は **文脈の取り違え**です。例えば「今週」が今日を含むのか、「北海道」がどの地方を指すのか(このノートブックでは札幌周辺の「石狩・空知・後志地方」を既定にしています)。
Tool の docstring と、`ask` 関数のシステムプロンプトに書いた前提が、LLM の解釈を決めます。

## 9. 自分の Tool を書いてみる

関数を 1 つ書いて `TOOLS` に足すだけです。守ることは 3 つ:

1. 引数に **型ヒント** を付ける(`area: str` など)
2. **docstring** に「何をするか」と、`Args:` で各引数の説明を日本語で書く(LLM はこれを読んで使い方を決める)
3. 戻り値は辞書やリスト(JSON にできるもの)

例として「週間予報の中で、降水確率が最も低い日」を返す Tool を書いてみます。

In [ ]:
#@title 例: 自分の Tool を追加する
def find_driest_day(area: str) -> dict:
    """
    指定した都道府県の週間予報から、降水確率が最も低い日(外出・洗濯・作業に向く日)を探します。

    Args:
        area: 都道府県名(例: "京都府", "東京都")。「京都」のような略称も可。
    """
    wk = get_weekly_forecast(area)
    days = [d for d in wk["日別予報"] if d.get("降水確率(%)")]
    best = min(days, key=lambda d: int(d["降水確率(%)"]))
    return {"地域": wk["地域"], "最も降水確率が低い日": best, "発表": wk["発表"]}

MY_TOOLS = TOOLS + [find_driest_day]
answer = ask("京都で今週、洗濯物を外に干すのに一番いい日は?", tools=MY_TOOLS)

In [ ]:
#@title ここに自分の Tool を書く(空欄)
def my_tool(area: str) -> dict:
    """
    (この関数が何をするかを書く)

    Args:
        area: (引数の説明を書く)
    """
    # ここに処理を書く。get_weekly_forecast(area) などを組み合わせて OK
    return {"結果": "まだ実装されていません"}

# answer = ask("(質問を書く)", tools=TOOLS + [my_tool])

## 10. まとめと、次にできること

**今日見たこと**
- LLM 単体は最新の天気を知らないが、Tool(関数)を渡すと公式データを取りに行って答える
- Tool は Python 関数と docstring だけ。関数を足せば機能が増える
- LLM は Tool の出力を疑わない。検証すべきは Tool と元データ

**次にできること**
- Tool を増やす: 地震情報、海洋(潮位・波浪)、気象庁の他の公開データ
- 地点指定の予報: Open-Meteo などのオープン API を Tool にすれば、緯度経度・標高で予報が取れる(海外モデル ECMWF IFS / AIFS の比較も)
- チャット画面にする: Gradio を使うと、このノートブック内にチャット UI が出せる
- 有料 API に置き換える: Claude / GPT などの API なら、同じ Tool 定義で更に安定した呼び出しができる

**出典・参考**
- 気象庁ホームページ https://www.jma.go.jp/ (予報・警報・アメダスの公開 JSON)。利用規約: https://www.jma.go.jp/jma/kishou/info/coment.html
- Qwen Team (2025). Qwen3 Technical Report. arXiv:2505.09388 / https://huggingface.co/Qwen
- Hugging Face Transformers, Chat templates: tool use

*このノートブックは Claude(Anthropic)と共同で作成しました。リポジトリ: https://github.com/deepkick/jma-tool-use-weather*